# Creating an HDF5 data sample

This notebook demonstrates how to convert a HELENA equilibrium together with
its MISHKA/CASTOR stability calculations into a compact HDF5 file.

The resulting file contains

- metadata
- equilibrium input parameters
- HELENA equilibrium profiles
- MISHKA stability results
- CASTOR stability results

and is intended as the basic unit of the dataset.

In [1]:
from pathlib import Path
import h5py

from karhu_data_handling.convert_sample import convert_sample

In [2]:
sample_dir = Path("C:/Users/AMBAMANDA/Projects/karhu-data-handling/tests/data/HelenaRunner-0a8221be-38f7-4679-a937-566e6bf83d5a_scan_2")
output_file = Path("C:/Users/AMBAMANDA/Projects/karhu-data-handling/tests/data/sample.h5")
convert_sample(sample_dir, output_file)

from pathlib import Path

def directory_size(path):
    """Return the total size (bytes) of all files in a directory."""
    return sum(
        f.stat().st_size
        for f in Path(path).rglob("*")
        if f.is_file()
    )

raw_size = directory_size(sample_dir)
h5_size = output_file.stat().st_size

print(f"Raw directory: {raw_size/1024**2:.2f} MB")
print(f"HDF5 file:     {h5_size/1024**2:.2f} MB")
print(f"Compression:   {raw_size/h5_size:.1f}x")


Saved C:\Users\AMBAMANDA\Projects\karhu-data-handling\tests\data\sample.h5
Raw directory: 144.83 MB
HDF5 file:     2.01 MB
Compression:   72.1x


In [3]:
def print_tree(group, indent=0):
    """
    Recursively print the contents of an HDF5 file.
    """

    for key in group.keys():

        item = group[key]

        print("    " * indent + key)

        if isinstance(item, h5py.Group):
            print_tree(item, indent + 1)

In [4]:
with h5py.File(output_file) as h5:
    print_tree(h5)

castor
    n003
        growthrate
        iterations
        mpol
        ntor
        params
            helena_dir
            ntor
    n005
        growthrate
        iterations
        mpol
        ntor
        params
            helena_dir
            ntor
    n007
        growthrate
        iterations
        mpol
        ntor
        params
            helena_dir
            ntor
    n010
        growthrate
        iterations
        mpol
        ntor
        params
            helena_dir
            ntor
    n015
        growthrate
        iterations
        mpol
        ntor
        params
            helena_dir
            ntor
    n020
        growthrate
        iterations
        mpol
        ntor
        params
            helena_dir
            ntor
    n030
        growthrate
        iterations
        mpol
        ntor
        params
            helena_dir
            ntor
    n050
        growthrate
        iterations
        mpol
        ntor
        params
         

In [5]:
with h5py.File(output_file) as h5:

    print("Metadata")

    for key, value in h5["metadata"].items():
        print(f"{key:15s} : {value[()]}")

Metadata
creation_date   : b'2026-07-28T17:09:31.214361'


In [6]:
with h5py.File(output_file) as h5:

    profiles = h5["equilibrium"]["profiles"]

    print("Available profiles:")

    for name in profiles.keys():
        print(name, profiles[name].shape)

Available profiles:
CHI (513,)
CPSURF ()
CS (301,)
CURJ (301,)
DJ0 ()
DJE ()
DP0 ()
DPE ()
DQEC ()
DQS (300,)
DQS_1 ()
DRBPHI0 ()
DRBPHIE ()
EPS ()
GEM11 (153900,)
GEM12 (153900,)
GEM33 (153900,)
JS0 ()
NCHI ()
P0 (301,)
QS (301,)
RADIUS ()
RAXIS ()
RBPHI (301,)
VX (513,)
VY (513,)
XOUT (153900,)
YOUT (153900,)


In [7]:
with h5py.File(output_file) as h5:
    if "mishka" in h5:
        for mode in h5["mishka"]:
            g = h5["mishka"][mode]

            print(mode)
            print("  ntor       :", g["ntor"][()])
            print("  growthrate :", g["growthrate"][()])
            print("  iterations :", g["iterations"][()])

n003
  ntor       : 3
  growthrate : [5.790e-03 2.077e-10]
  iterations : 13
n005
  ntor       : 5
  growthrate : [1.297e-02 3.360e-11]
  iterations : 6
n007
  ntor       : 7
  growthrate : [1.915e-02 6.296e-11]
  iterations : 3
n010
  ntor       : 10
  growthrate : [2.402e-02 7.874e-11]
  iterations : 5
n015
  ntor       : 15
  growthrate : [2.637e-02 7.935e-11]
  iterations : 6
n020
  ntor       : 20
  growthrate : [2.518e-02 9.625e-11]
  iterations : 5
n030
  ntor       : 30
  growthrate : [1.963e-02 9.207e-11]
  iterations : 3
n050
  ntor       : 50
  growthrate : [ 9.749e-03 -4.736e-11]
  iterations : 9


In [8]:
with h5py.File(output_file) as h5:
    if "castor" in h5:
        for mode in h5["castor"]:
            g = h5["castor"][mode]

            print(mode)
            print("  ntor       :", g["ntor"][()])
            print("  growthrate :", g["growthrate"][()])

n003
  ntor       : 3
  growthrate : [9.875e-03 1.830e-05]
n005
  ntor       : 5
  growthrate : [ 1.730e-02 -7.422e-05]
n007
  ntor       : 7
  growthrate : [ 2.498e-02 -8.544e-05]
n010
  ntor       : 10
  growthrate : [ 2.473e-02 -7.965e-05]
n015
  ntor       : 15
  growthrate : [ 3.256e-02 -8.409e-05]
n020
  ntor       : 20
  growthrate : [ 2.088e-02 -4.489e-05]
n030
  ntor       : 30
  growthrate : [ 3.253e-02 -6.869e-05]
n050
  ntor       : 50
  growthrate : [ 0.009707  -0.0001061]


In [9]:
import numpy as np

def print_contents(group, indent=0):
    """
    Print the complete contents of an HDF5 file.
    """

    for key in group.keys():

        obj = group[key]

        if isinstance(obj, h5py.Group):

            print("    " * indent + f"{key}/")
            print_contents(obj, indent + 1)

        else:

            value = obj[()]

            if isinstance(value, bytes):
                value = value.decode()

            if isinstance(value, np.ndarray):
                if value.size > 8:
                    text = f"array(shape={value.shape})"
                else:
                    text = value

            else:
                text = value

            print("    " * indent + f"{key}: {text}")

In [10]:
with h5py.File(output_file) as h5:
    print_contents(h5)

castor/
    n003/
        growthrate: [9.875e-03 1.830e-05]
        iterations: 8
        mpol: 71
        ntor: 3
        params/
            helena_dir: /scratch/project_2009007/data_JET_castor/20251213/HelenaRunner-0a8221be-38f7-4679-a937-566e6bf83d5a_scan_2
            ntor: 3
    n005/
        growthrate: [ 1.730e-02 -7.422e-05]
        iterations: 15
        mpol: 71
        ntor: 5
        params/
            helena_dir: /scratch/project_2009007/data_JET_castor/20251213/HelenaRunner-0a8221be-38f7-4679-a937-566e6bf83d5a_scan_2
            ntor: 5
    n007/
        growthrate: [ 2.498e-02 -8.544e-05]
        iterations: 5
        mpol: 71
        ntor: 7
        params/
            helena_dir: /scratch/project_2009007/data_JET_castor/20251213/HelenaRunner-0a8221be-38f7-4679-a937-566e6bf83d5a_scan_2
            ntor: 7
    n010/
        growthrate: [ 2.473e-02 -7.965e-05]
        iterations: 2
        mpol: 71
        ntor: 10
        params/
            helena_dir: /scratch/projec

        params/
            helena_dir: /scratch/project_2009007/data_JET_castor/20251213/HelenaRunner-0a8221be-38f7-4679-a937-566e6bf83d5a_scan_2
            ntor: 50
equilibrium/
    inputs/
        ball/
        num/
            amix: 0.0
            errcur: 1e-11
            nchi: 513
            niter: 50
            nmesh: 100
            np: 129
            npcur: 33
            npmap: 513
            nr: 301
            nrcur: 150
            nrmap: 301
        phys/
            adaptit: 0.0
            ade: 5.186245190008602
            amain: 1.9865111113
            ate: 0.10003823223699031
            ati: 0.0
            b: 0.001
            bde: 2.6332093520428312
            betap: -1
            bkeep: False
            bsmodel: 5
            bsmultip: 1.0
            bte: 0.1
            bti: 0.0
            bvac: 1.698447509280181
            cde: 0.1
            coreneo: 0.1
            corep: 79177.94410850559
            cte: 1.5669242654204163
            cti: 1.5